# Visualizations

Plotly charts from categorized discovery records.
Run `notebook/analysis.ipynb` first so CSVs exist under `output/processed/`.

Paths are anchored at the project root (`simple/`), one level above this notebook.


In [2]:

%pip install -q -r {_requirements()}
%pip install -q "nbformat>=4.2.0"


zsh: parse error near `}'
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 0. Imports & paths


In [3]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import umap

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts" / "sentiment_analysis.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected scripts/sentiment_analysis.py). "
        f"cwd={here}"
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.categorize_records import EMBEDDING_MODEL

CATEGORY_ORDER = [
    "Workflow & Business Processes",
    "Case Management",
    "Data Management & Visibility",
    "Records & Document Management",
    "System Integration",
    "Reporting & Decision Support",
    "Communication & Collaboration",
    "Scheduling & Resource Management",
    "User Experience & Performance",
    "Training & Documentation",
]

PROCESSED = ROOT / "output" / "processed"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CATEGORIZED_CHALLENGES_CSV = PROCESSED / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED / "categorized_expectations.csv"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


/Users/lilscott/Code/IPS/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Load categorized records

Reads `categorized_challenges.csv` / `categorized_expectations.csv` written by analysis.
The semantic map cell re-embeds challenges (embeddings are not persisted).


In [4]:
missing = [p for p in (CATEGORIZED_CHALLENGES_CSV, CATEGORIZED_EXPECTATIONS_CSV) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Run notebook/analysis.ipynb through the save step first. Missing:\n  - "
        + "\n  - ".join(str(p) for p in missing)
    )

df = pd.read_csv(CATEGORIZED_CHALLENGES_CSV)
expectations_df = pd.read_csv(CATEGORIZED_EXPECTATIONS_CSV)

print(f"Challenges: {len(df)} | Expectations: {len(expectations_df)}")
print(f"Categories: {df['Category'].nunique()} | Focus groups: {df['focus_group'].nunique()}")


Challenges: 1151 | Expectations: 540
Categories: 10 | Focus groups: 20


## 1b. Reconcile categories → `final_category`

Confidence uses the **entropy** of the semantic similarity distribution
(`Category_Confidence_Norm`, 0–1: high = mass concentrated on one category). Purity
is made **chance/size-robust** — raw purity is trivially 1.0 for tiny clusters, so it
is shrunk toward the dominant category's base rate (`Cluster_Purity_Adj`) and gated by
a minimum cluster size. A record adopts its cluster's label only when the cluster is
large + pure enough *and* its adjusted purity beats the record's confidence.
`ARI` / `AMI` / `V-measure` report chance-corrected cluster ↔ category agreement.

In [5]:
def add_final_category(frame, min_purity=0.6, min_cluster_size=5, smoothing=5.0):
    """Reconcile per-record Category with the cluster's dominant label (Cluster_Label).

    Confidence = ENTROPY of the semantic distribution (Category_Confidence_Norm, 0-1;
    produced in categorize_dataframe). Purity is chance/size-robust: raw purity is
    trivially 1.0 for tiny clusters, so we shrink it toward the dominant category's
    base rate (empirical-Bayes) -> Cluster_Purity_Adj, and gate on a minimum cluster
    size. A record adopts its cluster's label only when the cluster is big + pure
    enough AND its adjusted purity beats the record's confidence (all on 0-1).
    """
    frame = frame.copy()

    size = frame.groupby("Cluster")["Category"].transform("size")
    matches = (frame["Category"] == frame["Cluster_Label"]).groupby(frame["Cluster"]).transform("sum")
    raw_purity = matches / size
    base_rate = frame["Cluster_Label"].map(frame["Category"].value_counts(normalize=True)).fillna(0.0)
    adj_purity = (matches + smoothing * base_rate) / (size + smoothing)  # shrink small clusters

    frame["Cluster_Size"] = size
    frame["Cluster_Purity"] = np.where(frame["Cluster"] == -1, np.nan, raw_purity)
    frame["Cluster_Purity_Adj"] = np.where(frame["Cluster"] == -1, np.nan, adj_purity)

    # Entropy-based confidence from categorize_dataframe; fall back to a rescale if absent.
    if "Category_Confidence_Norm" not in frame.columns:
        conf = frame["Category_Confidence"].clip(lower=0)
        hi = float(conf.quantile(0.95))
        frame["Category_Confidence_Norm"] = (conf / hi).clip(0, 1) if hi > 0 else 0.0

    frame["final_category"] = frame["Category"]
    take_cluster = (
        (frame["Cluster"] != -1)
        & (frame["Cluster_Size"] >= min_cluster_size)
        & (frame["Cluster_Purity_Adj"] >= min_purity)
        & (frame["Cluster_Purity_Adj"] > frame["Category_Confidence_Norm"])
    )
    frame.loc[take_cluster, "final_category"] = frame.loc[take_cluster, "Cluster_Label"]
    frame["Category_Reassigned"] = frame["final_category"] != frame["Category"]
    return frame


def cluster_agreement(frame):
    """Chance-corrected agreement between clusters and categories (non-noise rows)."""
    from sklearn.metrics import (
        adjusted_rand_score, adjusted_mutual_info_score, homogeneity_completeness_v_measure,
    )
    m = frame["Cluster"] != -1
    if int(m.sum()) < 2:
        return {}
    a, c = frame.loc[m, "Category"], frame.loc[m, "Cluster"]
    return {
        "ARI": round(adjusted_rand_score(a, c), 3),
        "AMI": round(adjusted_mutual_info_score(a, c), 3),
        "V_measure": round(homogeneity_completeness_v_measure(a, c)[2], 3),
    }


df = add_final_category(df)
expectations_df = add_final_category(expectations_df)

for label, frame in [("Challenges", df), ("Expectations", expectations_df)]:
    n = int(frame["Category_Reassigned"].sum())
    print(f"{label}: {n}/{len(frame)} took the cluster label ({100 * n / len(frame):.1f}%) "
          f"| cluster vs category {cluster_agreement(frame)}")

Challenges: 18/1151 took the cluster label (1.6%) | cluster vs category {'ARI': 0.069, 'AMI': np.float64(0.183), 'V_measure': np.float64(0.218)}
Expectations: 14/540 took the cluster label (2.6%) | cluster vs category {'ARI': 0.086, 'AMI': np.float64(0.195), 'V_measure': np.float64(0.246)}


In [6]:
# Adjusted purity vs entropy confidence — matching 0-1 scales (challenges).
print("Cluster vs Category (chance-corrected):", cluster_agreement(df))

mask = df["Cluster"] != -1
purity_summary = (
    df[mask]
    .groupby(["Cluster", "Cluster_Label"])
    .agg(
        records=("Category", "size"),
        purity=("Cluster_Purity", "first"),
        purity_adj=("Cluster_Purity_Adj", "first"),
        mean_confidence=("Category_Confidence_Norm", "mean"),
        reassigned=("Category_Reassigned", "sum"),
    )
    .reset_index()
    .sort_values("purity_adj", ascending=False)
)
display(purity_summary.round(3))

fig_pc = px.scatter(
    df[mask], x="Category_Confidence_Norm", y="Cluster_Purity_Adj",
    color="Category_Reassigned", size="Cluster_Size",
    hover_data=["focus_group", "Category", "final_category", "Cluster_Size", "Cluster_Purity"],
    title="Adjusted cluster purity vs entropy confidence (both 0–1)",
    color_discrete_map={True: "#E76F51", False: "#2A9D8F"},
    labels={"Category_Reassigned": "Took cluster label"},
)
fig_pc.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="#94A3B8"))
fig_pc.add_hline(y=0.6, line_dash="dot", line_color="#94A3B8", annotation_text="purity floor 0.6")
fig_pc.update_layout(
    height=560, width=860, template="plotly_white",
    xaxis=dict(range=[0, 1], title="Category confidence (entropy, 0–1)"),
    yaxis=dict(range=[0, 1], title="Cluster purity (adjusted, 0–1)"),
)
fig_pc.show()

Cluster vs Category (chance-corrected): {'ARI': 0.069, 'AMI': np.float64(0.183), 'V_measure': np.float64(0.218)}


,Cluster,Cluster_Label,records,purity,purity_adj,mean_confidence,reassigned
17,17,Case Management,20,0.850,0.728,0.379,3
15,15,Scheduling & Resource Management,12,1.000,0.726,0.787,0
4,4,Case Management,56,0.750,0.708,0.533,13
18,18,Case Management,18,0.833,0.704,0.501,2
8,8,User Experience & Performance,52,0.596,0.558,0.423,0
2,2,User Experience & Performance,18,0.667,0.556,0.508,0
16,16,Case Management,20,0.600,0.528,0.423,0
7,7,User Experience & Performance,14,0.643,0.516,0.308,0
21,21,Records & Document Management,52,0.538,0.506,0.424,0
0,0,System Integration,16,0.625,0.496,0.294,0


## 2. Category counts


In [7]:
category_counts = (
    df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_cat = px.bar(
    category_counts, x="Count", y="Category", orientation="h",
    title="Pain point categories", color="Count", color_continuous_scale="Blues",
)
fig_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_cat.show()

expectation_counts = (
    expectations_df.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Count"})
    .sort_values("Count", ascending=True)
)
fig_exp_cat = px.bar(
    expectation_counts, x="Count", y="Category", orientation="h",
    title="Expectation categories", color="Count", color_continuous_scale="Teal",
)
fig_exp_cat.update_layout(showlegend=False, height=500, width=1200, template="plotly_white")
fig_exp_cat.show()


## 4. Challenges vs expectations by theme


In [8]:
ch_c = df["Category"].value_counts()
ex_c = expectations_df["Category"].value_counts()
cats = sorted(set(ch_c.index) | set(ex_c.index), key=lambda c: -(ch_c.get(c, 0) + ex_c.get(c, 0)))
plot_cmp = pd.DataFrame({
    "Category": cats,
    "Challenges": [int(ch_c.get(c, 0)) for c in cats],
    "Expectations": [int(ex_c.get(c, 0)) for c in cats],
}).melt(id_vars="Category", var_name="Type", value_name="Count")
fig_cmp = px.bar(
    plot_cmp, x="Count", y="Category", color="Type", barmode="group", orientation="h",
    title="Challenges vs expectations by theme",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
)
fig_cmp.update_layout(height=520, template="plotly_white", yaxis=dict(categoryorder="total ascending"))
fig_cmp.show()


## 5. Top focus groups


In [9]:
top_n = 20
top = df.groupby("focus_group").size().sort_values(ascending=False).head(top_n)
exp = expectations_df.groupby("focus_group").size()
plot_top = pd.DataFrame({
    "Focus Group": top.index.tolist(),
    "Challenges": top.values.astype(int),
    "Expectations": [int(exp.get(g, 0)) for g in top.index],
}).melt(id_vars="Focus Group", var_name="Kind", value_name="Count")
fig_top = px.bar(
    plot_top,
    x="Focus Group",
    y="Count",
    color="Kind",
    barmode="group",
    title=f"Top {top_n} focus groups",
    color_discrete_map={"Challenges": "#E76F51", "Expectations": "#2A9D8F"},
    category_orders={"Focus Group": top.index.tolist()},
)
fig_top.update_layout(
    height=560,
    width=1000,
    template="plotly_white",
    xaxis=dict(tickangle=-40),
    margin=dict(b=140),
)
fig_top.show()


## 6. Category heatmaps

- **Dense cluster × category** — each cluster is its own row (id + dominant label),
  columns are the per-record `Category`; diagonal-heavy rows are pure clusters.
- **Interactive focus group × final category** — click-free drill-down further down.

In [16]:
heatmap_df = pd.crosstab(df["focus_group"], df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Groups", y="Final category", color="Count"),
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Challenges: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [17]:
heatmap_df = pd.crosstab(df["focus_group"], df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Focus Groups", y="Final category", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Challenges: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

### Department × final category

Same heatmap grouped by department instead of focus group.

In [12]:
heatmap_df = pd.crosstab(df["department"], df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Department", y="Final category", color="Count"),
    color_continuous_scale="YlOrRd", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Challenges: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [13]:
heatmap_df = pd.crosstab(expectations_df["department"], expectations_df["final_category"])
department_order = heatmap_df.sum(axis=1).sort_values(ascending=False).index
category_order = heatmap_df.sum(axis=0).sort_values(ascending=False).index
heatmap_df = heatmap_df.loc[department_order, category_order].T
fig_heatmap = px.imshow(
    heatmap_df,
    labels=dict(x="Department", y="Final category", color="Count"),
    color_continuous_scale="Teal", text_auto=True, aspect="auto",
)
fig_heatmap.update_layout(
    title="Expectations: department × final category",
    height=700, width=1000, xaxis_tickangle=-45, template="plotly_white",
)
fig_heatmap.show()

In [25]:
# Truly clickable heatmap for the Cursor / VS Code notebook renderer.
# plotly FigureWidget.on_click does NOT fire here, but ipywidgets Button.on_click does,
# so the heatmap is drawn as a grid of colored buttons — click a cell to list its records.
import matplotlib
from ipywidgets import Button, GridBox, Layout, Output, HTML, VBox
from IPython.display import display, clear_output


def clickable_heatmap(frame, text_col, noun, cmap_name, out_html, category_col="final_category"):
    heat = pd.crosstab(frame["focus_group"], frame[category_col])
    heat = heat.reindex(columns=[c for c in CATEGORY_ORDER if c in heat.columns])
    heat = heat.loc[heat.sum(axis=1).sort_values(ascending=False).index]
    focus_groups = heat.index.tolist()   # grid columns
    categories = heat.columns.tolist()   # grid rows
    vmax = max(1, int(heat.values.max()))
    cmap = matplotlib.colormaps[cmap_name]

    def cell_color(v):
        r, g, b, _ = cmap(v / vmax)
        return f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"

    # Static Plotly copy for export/sharing (same orientation: categories on Y).
    zt = heat.T
    go.Figure(go.Heatmap(
        z=zt.values, x=focus_groups, y=categories, colorscale=cmap_name,
        text=zt.values, texttemplate="%{text}", xgap=1, ygap=1,
    )).update_layout(
        title=f"{noun} volume: focus group × final category",
        template="plotly_white", height=600, width=1100,
        xaxis=dict(tickangle=0), yaxis=dict(autorange="reversed"),
    ).write_html(out_html, include_plotlyjs="cdn")
    print(f"Saved {out_html}")

    detail_cols = [
        c for c in ["focus_group", text_col, "final_category", "Category",
                    "Category_Confidence", "Cluster_Label", "Cluster_Purity"]
        if c in frame.columns
    ]
    out = Output()

    def show_records(fg, cat):
        sub = frame[(frame["focus_group"] == fg) & (frame[category_col] == cat)][detail_cols].drop_duplicates()
        if "Category_Confidence" in sub.columns:
            sub = sub.sort_values("Category_Confidence", ascending=False)
        with out:
            clear_output(wait=True)
            print(f"{len(sub)} {noun.lower()} records · {fg} × {cat}")
            display(sub if not sub.empty else "No records for this selection.")

    col_w, label_w, header_h = 48, 230, 70
    cells = [HTML("")]  # top-left corner
    for fg in focus_groups:  # horizontal, wrapped column headers
        cells.append(HTML(
            f"<div title='{fg}' style='height:{header_h}px;width:{col_w}px;display:flex;"
            f"align-items:flex-end;justify-content:center'><span style='font-size:10px;"
            f"font-weight:600;line-height:1.1;text-align:center;white-space:normal;"
            f"overflow-wrap:anywhere'>{fg}</span></div>"))
    for cat in categories:
        cells.append(HTML(
            f"<div style='height:34px;display:flex;align-items:center;justify-content:flex-end;"
            f"padding-right:6px;font-size:11px;font-weight:600'>{cat}</div>"))
        for fg in focus_groups:
            v = int(heat.loc[fg, cat])
            b = Button(description=(str(v) if v else ""),
                       tooltip=f"{cat} × {fg}: {v} — click to view",
                       layout=Layout(width=f"{col_w}px", height="34px", margin="0px"))
            b.style.button_color = cell_color(v)
            if v / vmax >= 0.55:
                b.style.text_color = "white"
            b.on_click(lambda _b, fg=fg, cat=cat: show_records(fg, cat))
            cells.append(b)

    grid = GridBox(cells, layout=Layout(
        grid_template_columns=f"{label_w}px " + f"{col_w}px " * len(focus_groups),
        grid_gap="1px"))
    display(VBox([
        HTML(f"<h4 style='margin:4px 0'>{noun} volume: focus group × final category "
             f"<span style='font-weight:400;color:#64748b'>(click a cell to view records)</span></h4>"),
        grid, out]))


# Challenges
clickable_heatmap(
    df, CHALLENGE_TEXT_COL, "Challenge", "YlOrRd",
    FIG_DIR / "heatmap_focus_category.html",
)

Saved /Users/lilscott/Code/IPS/output/figures/heatmap_focus_category.html


In [15]:
# Expectations (same clickable heatmap)
clickable_heatmap(
    expectations_df, EXPECTATION_TEXT_COL, "Expectation", "GnBu",
    FIG_DIR / "heatmap_focus_category_expectations.html",
)

Saved /Users/lilscott/Code/IPS/output/figures/heatmap_focus_category_expectations.html
